# F1 Lap Dataset

Builds a lap-level dataset from Jolpica F1 CSV data (1950-2025) enriched with
FastF1 telemetry (2018+). Also computes Elo-MMR ratings for constructors and
engine suppliers across the full history of the championship.

In [11]:
import pandas as pd
import fastf1
import os
from elo_mmr_py import Contest, rate
from collections import defaultdict
from fastf1.exceptions import RateLimitExceededError
import time

FF1_CACHE_PATH = "./cache/jolpica_fastf1_cache.csv"
DESIRED_COLS = ["Driver", "LapNumber", "Compound", "TyreLife", "SpeedST", 
                "TrackStatus", "AirTemp", "TrackTemp", "WindSpeed", "WindDirection",
                "Sector1Time", "Sector2Time", "Sector3Time", "Rainfall"]

WEATHER_COLS = [
    "Rainfall", "AirTemp", "TrackTemp", "WindSpeed", "WindDirection"
]

session_csv = pd.read_csv("./datasets/formula_one_session.csv")
session_entry_csv = pd.read_csv("./datasets/formula_one_sessionentry.csv")
laps_csv = pd.read_csv("./datasets/formula_one_lap.csv")
round_csv = pd.read_csv("./datasets/formula_one_round.csv")
round_entry_csv = pd.read_csv("./datasets/formula_one_roundentry.csv")
team_driver_csv = pd.read_csv("./datasets/formula_one_teamdriver.csv")
team_csv = pd.read_csv("./datasets/formula_one_team.csv")
driver_csv = pd.read_csv("./datasets/formula_one_driver.csv")
pit_stop_csv = pd.read_csv("./datasets/formula_one_pitstop.csv")
engine_map_csv = pd.read_csv("./datasets/engine_map.csv")

race_sessions = session_csv[session_csv["type"] == "R"].copy()
race_sessions["year"] = pd.to_datetime(race_sessions["timestamp"], utc=True).dt.year
session_to_round_id = session_csv.set_index("id")["round_id"]
round_id_to_number = round_csv.set_index("id")["number"]

fastf1.Cache.enable_cache("./cache")
fastf1.Cache.offline_mode(True)  # offline. rely on local fastf1 cache only

needed = (
    race_sessions[(race_sessions["year"] >= 2018) & (race_sessions["year"] <= 2026)][["id", "year"]].copy()
    .assign(round_number=lambda d: d["id"].map(session_to_round_id).map(round_id_to_number))
    [["year", "round_number"]].drop_duplicates().dropna()
)

if os.path.exists(FF1_CACHE_PATH):
    ff1_cache = pd.read_csv(FF1_CACHE_PATH)
    done = set(zip(ff1_cache["year"], ff1_cache["round_number"]))
    missing_cols = set(DESIRED_COLS) - set(ff1_cache.columns)
    if missing_cols:
        print(f"FastF1 cache missing {missing_cols} -- delete cache to rebuild with all columns.")
        exit(0)
else:
    ff1_cache = pd.DataFrame()
    done = set()
    

to_fetch = needed[~needed.apply(lambda r: (r["year"], r["round_number"]) in done, axis=1)]
print(f"FastF1: {len(done)} races cached, {len(to_fetch)} to fetch")


for _, row in to_fetch.iterrows():
    year, rnd=int(row["year"]), int(row["round_number"])
    try:
        session = fastf1.get_session(year, rnd, "R")
        session.load(laps=True, telemetry=False, weather=True, messages=False)

        cols = [c for c in DESIRED_COLS if c in session.laps.columns]
        laps = session.laps[cols].copy()
        for s_col in ["Sector1Time", "Sector2Time", "Sector3Time"]:
            if s_col in laps.columns:
                laps[s_col] = laps[s_col].dt.total_seconds()
        weather_sorted = session.weather_data.sort_values("Time")
        for lap in laps.itertuples():
            lap_time = session.laps.loc[lap.Index, "LapStartTime"]
            matching = weather_sorted[weather_sorted["Time"] <= lap_time]
            if not matching.empty:
                w_row = matching.iloc[-1]
                for w_col in WEATHER_COLS:
                    laps.at[lap.Index, w_col] = w_row[w_col]

        laps["year"] = year
        laps["round_number"] = rnd
        laps.to_csv(FF1_CACHE_PATH, mode="a", header=not os.path.exists(FF1_CACHE_PATH), index=False)
        done.add((year, rnd))
        print(f"fetched {year} R{rnd}")
    except RateLimitExceededError as e:
        print(f"rate limit exceeded at {year} R{rnd} -- stopping: {e}")
        time.sleep(60 * 80)
    except Exception as e:
        print(f"skipped {year} R{rnd}: {e}")

ff1_cache = pd.read_csv(FF1_CACHE_PATH) if os.path.exists(FF1_CACHE_PATH) else pd.DataFrame()
print(f"FastF1 cache: {len(ff1_cache) // 1000}k rows")

CORNERS_CACHE_PATH = "./cache/circuit_corners.csv"
corners_csv = pd.read_csv(CORNERS_CACHE_PATH) if os.path.exists(CORNERS_CACHE_PATH) else pd.DataFrame()
print(f"Corners cache: {len(corners_csv)} circuits")


/tmp/ipykernel_92333/4128764045.py:44: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  ff1_cache = pd.read_csv(FF1_CACHE_PATH)
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req

FastF1: 175 races cached, 22 to fetch
skipped 2026 R3: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R4: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R5: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R6: The data you are trying to access has not been loaded yet. See `Session.load`


req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
logger      WARNING 	Failed to load weather data!
core           INFO 	Finished loading data for 0 drivers: []
core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger  

skipped 2026 R7: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R8: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R9: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R10: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R11: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R12: The data you are trying to access has not been loaded yet. See `Session.load`


logger      WARNING 	Failed to load weather data!
core           INFO 	Finished loading data for 0 drivers: []
core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_

skipped 2026 R13: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R14: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R15: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R16: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R17: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R18: The data you are trying to access has not been loaded yet. See `Session.load`


req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
logger      WARNING 	Failed to load total lap count!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Cannot load lap times for first lap from Ergast. Timing data is not available for this session.
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
logger      WARNING 	Failed 

skipped 2026 R19: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R20: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R21: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R22: The data you are trying to access has not been loaded yet. See `Session.load`
skipped 2026 R23: Invalid round: 23
skipped 2026 R24: Invalid round: 24


/tmp/ipykernel_92333/4128764045.py:90: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  ff1_cache = pd.read_csv(FF1_CACHE_PATH) if os.path.exists(FF1_CACHE_PATH) else pd.DataFrame()


FastF1 cache: 191k rows
Corners cache: 173 circuits


## Lap Dataset

Race laps with metadata, cumulative times, on-track gaps, pit markers, and FastF1 tyre/weather/sector data joined by (year, round, driver, lap number).

In [12]:
def parse_flag(status):
    s = str(status) if pd.notna(status) else ""
    if "5" in s: return "RED"
    if "4" in s: return "SC"
    if "6" in s or "7" in s: return "VSC"
    if "2" in s or "3" in s: return "YELLOW"
    return "CLEAR"

race_entries = session_entry_csv[session_entry_csv["session_id"].isin(race_sessions["id"])]
race_laps = laps_csv[laps_csv["session_entry_id"].isin(race_entries["id"])].copy()

race_laps["session_id"] = race_laps["session_entry_id"].map(race_entries.set_index("id")["session_id"])
race_laps["year"] = race_laps["session_id"].map(race_sessions.set_index("id")["year"])
race_laps["round_number"] = race_laps["session_id"].map(session_to_round_id).map(round_id_to_number)
race_laps["round_entry_id"] = race_laps["session_entry_id"].map(race_entries.set_index("id")["round_entry_id"])
race_laps["team_driver_id"] = race_laps["round_entry_id"].map(round_entry_csv.set_index("id")["team_driver_id"])
race_laps["driver_id"] = race_laps["team_driver_id"].map(team_driver_csv.set_index("id")["driver_id"])
race_laps["driver"] = race_laps["driver_id"].map(driver_csv.set_index("id")["reference"])

race_laps["lap_time"] = pd.to_timedelta(race_laps["time"])
race_laps = race_laps.sort_values(["session_id", "driver", "number"])
race_laps["cumulative_time"] = race_laps.groupby(["session_id", "driver"])["lap_time"].cumsum()

race_laps = race_laps.sort_values(["session_id", "number", "position"])
race_laps["gap_from_leader"] = (
    race_laps["cumulative_time"]
    - race_laps.groupby(["session_id", "number"])["cumulative_time"].transform("min")
).dt.total_seconds()
race_laps["gap_from_ahead"] = (
    race_laps.groupby(["session_id", "number"])["cumulative_time"].diff().dt.total_seconds()
)
race_laps["gap_from_behind"] = (
    -race_laps.groupby(["session_id", "number"])["cumulative_time"].diff(-1).dt.total_seconds()
)

# pit stop markers. 0 = none. 1 = pit_in. 2 = out. NaN = no data
race_pit_stops = pit_stop_csv[pit_stop_csv["session_entry_id"].isin(race_entries["id"])]
sessions_with_pit_data = set(race_pit_stops["session_entry_id"].map(race_entries.set_index("id")["session_id"]))
pit_in_lap_ids = set(race_pit_stops["lap_id"])
race_laps["pit_lap"] = 0
race_laps.loc[race_laps["id"].isin(pit_in_lap_ids), "pit_lap"] = 1
pit_in_rows = race_laps[race_laps["pit_lap"] == 1][["session_entry_id", "number"]]
exit_keys = set(
    pit_in_rows["session_entry_id"].astype(int).astype(str) + "_" +
    (pit_in_rows["number"] + 1).astype(int).astype(str)
)
all_keys = race_laps["session_entry_id"].astype(int).astype(str) + "_" + race_laps["number"].astype(int).astype(str)
race_laps.loc[(race_laps["pit_lap"] == 0) & all_keys.isin(exit_keys), "pit_lap"] = 2
race_laps.loc[~race_laps["session_id"].isin(sessions_with_pit_data), "pit_lap"] = float("nan")

# FastF1 join
if not ff1_cache.empty:
    code_to_ref = driver_csv.dropna(subset=["abbreviation"]).set_index("abbreviation")["reference"].to_dict()
    ff1 = ff1_cache.copy()
    ff1["driver"] = ff1["Driver"].map(code_to_ref)
    ff1["flag"] = ff1["TrackStatus"].apply(parse_flag)
    ff1 = ff1.sort_values(["year", "round_number", "driver", "LapNumber"])
    ff1["_flag_group"] = ff1.groupby(["year", "round_number", "driver"])["flag"].transform(
        lambda x: (x != x.shift()).cumsum()
    )
    ff1["laps_since_flag_change"] = ff1.groupby(
        ["year", "round_number", "driver", "_flag_group"]
    ).cumcount()
    ff1.drop(columns=["_flag_group"], inplace=True)
    ff1.rename(columns={
        "LapNumber": "number",
        "Compound": "compound",
        "TyreLife": "tyre_age",
        "SpeedST": "speed_trap",
        "AirTemp": "air_temp",
        "TrackTemp": "track_temp",
        "WindSpeed": "wind_speed",
        "WindDirection": "wind_direction",
        "Sector1Time": "sector1_time",
        "Sector2Time": "sector2_time",
        "Sector3Time": "sector3_time",
    }, inplace=True)
    join_cols = ["year", "round_number", "driver", "number",
                 "compound", "tyre_age", "flag", "laps_since_flag_change"]
    for col in ["speed_trap", "air_temp", "track_temp", "wind_speed", "wind_direction",
                "sector1_time", "sector2_time", "sector3_time"]:
        if col in ff1.columns:
            join_cols.append(col)
    ff1["number"] = ff1["number"].astype(float)
    race_laps = race_laps.merge(ff1[join_cols], on=["year", "round_number", "driver", "number"], how="left")

if not corners_csv.empty:
    race_laps = race_laps.merge(
        corners_csv[["year", "round_number", "corners", "slow_corners",
                    "medium_corners", "high_speed_corners"]],
        on = ["year", "round_number"], how="left"
    )


## Race Results

`all_race_results`: `{(season, round): DataFrame[constructorName, position]}` — input to all three ELO computations.

In [13]:
session_to_year = race_sessions.set_index("id")["year"]
round_entry_to_team_driver = round_entry_csv.set_index("id")["team_driver_id"]
team_driver_to_team = team_driver_csv.set_index("id")["team_id"]
team_id_to_name = team_csv.set_index("id")["name"]

race_session_entries = session_entry_csv[session_entry_csv["session_id"].isin(race_sessions["id"])].copy()
race_session_entries["year"] = race_session_entries["session_id"].map(session_to_year)
race_session_entries["round_id"] = race_session_entries["session_id"].map(session_to_round_id)
race_session_entries["round_number"] = race_session_entries["round_id"].map(round_id_to_number)
race_session_entries["team_driver_id"] = race_session_entries["round_entry_id"].map(round_entry_to_team_driver)
race_session_entries["team_id"] = race_session_entries["team_driver_id"].map(team_driver_to_team)
race_session_entries["constructor_name"] = race_session_entries["team_id"].map(team_id_to_name)

elo_base = race_session_entries[["year", "round_number", "constructor_name", "position"]].dropna(
    subset = ["constructor_name", "year", "round_number"]
).copy()
elo_base["year"] = elo_base["year"].astype(int)
elo_base["round_number"] = elo_base["round_number"].astype(int)

all_race_results: dict[tuple[int, int], "pd.DataFrame"] = {
    (int(s), int(r)): g[["constructor_name", "position"]].reset_index(drop=True)
    for (s, r), g in elo_base.groupby(["year", "round_number"])
}

print(f"Loaded {len(all_race_results)} races ({elo_base['year'].min()}-{elo_base['year'].max()})")

Loaded 1149 races (1950-2025)


## FastF1 ELO Supplement

For races in `ff1Cache` absent from Jolpica (e.g. current-season rounds), loads FastF1 results and populates `all_race_results`, `extra_driver_base_rows`, and `_ff1_extra_sessions`.

In [14]:
abbr_to_ref_elo = (
    driver_csv.dropna(subset=["abbreviation"])
    .set_index("abbreviation")["reference"]
    .to_dict()
)

existing_elo_pairs = set(all_race_results.keys())
ff1_pairs_all = (
    set(zip(ff1_cache["year"].astype(int), ff1_cache["round_number"].astype(int)))
    if not ff1_cache.empty else set()
)
missing_elo_pairs = sorted(ff1_pairs_all - existing_elo_pairs)
print(f"Races to supplement ELO inputs with: {missing_elo_pairs}")

extra_driver_base_rows: list[dict] = []
_ff1_extra_sessions: dict={}

for (yr, rnd) in missing_elo_pairs:
    print(f"Loading {yr} R{rnd} for ELO ...")
    try:
        session = fastf1.get_session(yr, rnd, "R")
        # laps=True: session reused in model supplement cell
        session.load(laps=True, telemetry=False, weather=False, messages=False)
    except Exception as exc:
        print(f"Failed: {exc}")
        continue

    results = session.results
    if results is None or results.empty:
        print("No results available")
        continue

    _ff1_extra_sessions[(yr, rnd)] = session

    ctor_rows: list[dict] = []
    for _, row in results.iterrows():
        team = row.get("TeamName", "")
        abbr = row.get("Abbreviation", "")
        try:
            pos = float(row.get("Position", float("nan")))
        except (ValueError, TypeError):
            pos = float("nan")

        if pd.notna(pos) and team:
            ctor_rows.append({"constructor_name": str(team), "position": pos})

        ref = abbr_to_ref_elo.get(str(abbr))
        if ref and pd.notna(pos):
            extra_driver_base_rows.append({
                "year": yr, "round_number": rnd,
                "driver": ref, "position": pos,
            })

    if ctor_rows:
        all_race_results[(yr, rnd)] = pd.DataFrame(ctor_rows)
        print(f"+{len(ctor_rows)} constructor, " +
              f"+{sum(1 for r in extra_driver_base_rows if r['year'] == yr and r['round_number'] == rnd)} driver entries")
    else:
        print("No usable results")

sorted_results = sorted(all_race_results.items())
print(f"\nall_race_results: {len(all_race_results)} races " + 
      f"({min(k[0] for k in all_race_results)}-{max(k[0] for k in all_race_results)})")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Races to supplement ELO inputs with: [(2026, 1), (2026, 2)]
Loading 2026 R1 for ELO ...


core        WARNING 	No lap data for driver 81
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 81)
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '16', '44', '1', '3', '87', '41', '5', '10', '31', '23', '30', '43', '55', '11', '18', '14', '77', '6', '81', '27']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


+22 constructor, +22 driver entries
Loading 2026 R2 for ELO ...


core        WARNING 	Driver 12 completed the race distance 00:00.022000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '87', '10', '30', '6', '55', '43', '27', '41', '77', '31', '11', '3', '14', '18', '81', '1', '5', '23']


+22 constructor, +22 driver entries

all_race_results: 1151 races (1950-2026)


## Constructor ELO

Rates every constructor across the full 1950-2025 championship using Elo-MMR.
Each race is a contest; constructors are ranked by their best (lowest) finishing
position.

In [15]:
import json, tempfile

def split_into_eras(sorted_results, reset_years):
    eras, current, ptr = [], [], 0
    for (season, _round), df in sorted_results:
        while ptr < len(reset_years) and season >= reset_years[ptr]:
            eras.append(current); current = []; ptr += 1
        current.append(((season, _round), df))
    eras.append(current)
    return eras

def _build_standings(sorted_items):
    standings, rank, prev, band = [], 0, None, []
    def flush():
        nonlocal rank
        lo, hi = rank, rank + len(band) - 1
        for name in band:
            standings.append((name, lo, hi))
        rank = hi + 1
    for name, pos in sorted_items:
        if pos != prev:
            flush()
            band, prev = [name], pos
        else:
            band.append(name)
    flush()
    return standings

def _half_reset_checkpoint(src_path, dst_path, keep_names, mu_noob=1500.0, sig_noob=350.0):
    with open(src_path) as f:
        data = json.load(f)
    data = {k: v for k, v in data.items() if k in keep_names}
    for player in data.values():
        prev_mu = player["approx_posterior"]["mu"]
        new_mu = (prev_mu + mu_noob) / 2
        player["normal_factor"]["mu"] = new_mu
        player["normal_factor"]["sig"] = sig_noob
        player["logistic_factors"] = []
        player["event_history"] = []
        player["approx_posterior"]["mu"] = new_mu
        player["approx_posterior"]["sig"] = sig_noob
        player["update_time"] = 0
        player["delta_time"] = 0
    with open(dst_path, "w") as f:
        json.dump(data, f)

def run_elo_mmr(sorted_results, reset_years, build_standings_fn, history_key):
    # run Elo-MMR across eras; each era starts from a half-reset of the previous era's ratings
    era_splits = split_into_eras(sorted_results, reset_years)
    history_rows = []
    all_races: dict = defaultdict(list)
    final_players: dict = {}

    with tempfile.TemporaryDirectory() as _tmp:
        prev_ckpt = None
        for era_i, era_items in enumerate(era_splits):
            if not era_items:
                continue
            contests = []
            ev_races: dict = defaultdict(list)

            for (season, _round), race_df in era_items:
                standings = build_standings_fn(season, race_df)
                if not standings:
                    continue
                names = [n for n, _, _ in standings]
                contests.append(Contest(standings=standings))
                for name in names:
                    ev_races[name].append((season, _round))
                    all_races[name].append((season, _round))

            if not contests:
                continue

            load_ckpt = None
            if prev_ckpt is not None:
                load_ckpt = os.path.join(_tmp, f"{history_key}_{era_i}_init.json")
                _half_reset_checkpoint(prev_ckpt, load_ckpt, keep_names=set(ev_races.keys()))

            save_ckpt = os.path.join(_tmp, f"{history_key}_{era_i}.json")
            players = rate(contests, load_checkpoint=load_ckpt, save_checkpoint=save_ckpt)
            prev_ckpt = save_ckpt

            for name, player in players.items():
                if name not in ev_races:
                    continue
                for (s, r), event in zip(ev_races[name], player.events):
                    history_rows.append({
                        "season": s, "round": r, history_key: name,
                        "rating_mu": round(event.rating_mu, 2),
                        "rating_sig": round(event.rating_sig, 2),
                        "perf_score": round(event.perf_score, 2),
                        "place": event.place,
                    })
                final_players[name] = player

    return history_rows, all_races, final_players

CONSTRUCTOR_RESET_YEARS = [2014, 2022, 2026] # turbo hybrid / cost-cap era / new aero

def _ctor_standings(season, race_df):
    pos = pd.to_numeric(race_df["position"], errors="coerce").fillna(999)
    best = pos.groupby(race_df["constructor_name"]).min().sort_values()
    if best.empty:
        return []
    return _build_standings(best.items())

ctor_history_rows, all_races_by_name, final_players = run_elo_mmr(
    sorted_results, CONSTRUCTOR_RESET_YEARS, _ctor_standings, "constructor"
)

constructor_elo_history_df = (
    pd.DataFrame(ctor_history_rows)
    .sort_values(["season", "round", "constructor"])
    .reset_index(drop=True)
)

best_perf_by_name = (
    constructor_elo_history_df.groupby("constructor")["perf_score"].max().to_dict()
)

records = []
for name, player in final_players.items():
    ev_races = all_races_by_name[name]
    records.append({
        "Constructor": name,
        "Rating": round(player.rating, 1),
        "Rating mu": round(player.events[-1].rating_mu, 1) if player.events else None,
        "Rating sig": round(player.events[-1].rating_sig, 1) if player.events else None,
        "Races": len(ev_races),
        "Best perf": round(best_perf_by_name.get(name, 0), 1),
        "Last season": ev_races[-1][0] if ev_races else None,
    })

elo_summary_df = (
    pd.DataFrame(records)
    .sort_values("Rating", ascending=False)
    .reset_index(drop=True)
)
elo_summary_df.index += 1
elo_summary_df.index.name = "Rank"

n_eras = len([e for e in split_into_eras(sorted_results, CONSTRUCTOR_RESET_YEARS) if e])
print(f"Constructor ELO: {len(sorted_results)} races across {n_eras} eras "
      f"(resets at {CONSTRUCTOR_RESET_YEARS})")
print(f"Constructor ELO history: {len(constructor_elo_history_df):,} rows "
      f"({constructor_elo_history_df['season'].min()}-{constructor_elo_history_df['season'].max()})")
display(elo_summary_df.head(15))

Constructor ELO: 1151 races across 4 eras (resets at [2014, 2022, 2026])
Constructor ELO history: 12,762 rows (1950-2026)


,Constructor,Rating,Rating mu,Rating sig,Races,Best perf,Last season
Rank,,,,,,,
1,Brawn,2301,2301,80,17,2557,2009
2,Red Bull,2222,2222,80,418,2633,2025
3,BAR,2207,2207,80,118,2426,2005
4,Mercedes,2163,2163,130,343,2606,2026
5,BMW Sauber,2161,2161,80,70,2560,2009
6,Toyota,2141,2141,80,140,2408,2009
7,Stewart,2121,2121,80,49,2489,1999
8,Renault,2119,2119,80,402,2595,2020
9,Racing Point,2090,2090,80,38,2471,2020


## Engine ELO

A year-aware ENGINE_MAP resolves each constructor to its engine supplier for that season; hyphenated historical names (e.g. "Lotus-Ford") are parsed automatically. Produces engine_elo_history_df.

In [16]:
ENGINE_RESET_YEARS = [1966, 1989, 2014, 2026] # 3L formula / turbo ban / V6 hybrid / 50 hybrid

ENGINE_MAP: dict[tuple[int, str], str] = {}
for _, row in engine_map_csv.iterrows():
    for yr in range(int(row["year_start"]), int(row["year_end"]) + 1):
        key = (yr, row["constructor"])
        if key not in ENGINE_MAP:
            ENGINE_MAP[key] = row["engine"]

def resolve_engine(season: int, constructor: str) -> str:
    key = (season, constructor)
    if key in ENGINE_MAP:
        return ENGINE_MAP[key]
    if "-" in constructor:
        return constructor.split("-", 1)[1].strip()
    return constructor

def _eng_standings(season, race_df):
    pos = pd.to_numeric(race_df["position"], errors="coerce").fillna(999)
    best = pos.groupby(race_df["constructor_name"]).min()
    eng_pos: dict = defaultdict(list)
    for ctor, p in best.items():
        eng_pos[resolve_engine(season, ctor)].append(p)
    avg = {e: sum(ps) / len(ps) for e, ps in eng_pos.items()}
    if not avg:
        return []
    return _build_standings(sorted(avg.items(), key=lambda x: x[1]))

engine_rows, all_races_e, final_players_e = run_elo_mmr(
    sorted_results, ENGINE_RESET_YEARS, _eng_standings, "engine"
)

engine_elo_history_df = (
    pd.DataFrame(engine_rows)
    .sort_values(["season", "round", "engine"])
    .reset_index(drop=True)
)

best_perf_e = engine_elo_history_df.groupby("engine")["perf_score"].max().to_dict()

engine_records = []
for eng_name, player in final_players_e.items():
    ev_races = all_races_e[eng_name]
    engine_records.append({
        "Engine": eng_name,
        "Rating": round(player.rating, 1),
        "Rating mu": round(player.events[-1].rating_mu, 1) if player.events else None,
        "Rating sig": round(player.events[-1].rating_sig, 1) if player.events else None,
        "Races": len(ev_races),
        "Best perf": round(best_perf_e.get(eng_name, 0), 1),
        "Last season": ev_races[-1][0] if ev_races else None,
    })

engine_elo_df = (
    pd.DataFrame(engine_records)
    .sort_values("Rating", ascending=False)
    .reset_index(drop=True)
)
engine_elo_df.index += 1
engine_elo_df.index.name = "Rank"

n_eras_e = len([e for e in split_into_eras(sorted_results, ENGINE_RESET_YEARS) if e])
print(f"Engine ELO: {len(sorted_results)} races across {n_eras_e} eras "
      f"(resets at {ENGINE_RESET_YEARS})")
print(f"Engine ELO history: {len(engine_elo_history_df):,} rows"
      f"({engine_elo_history_df['season'].min()}-{engine_elo_history_df['season'].max()})")
display(engine_elo_df.head(15))


Engine ELO: 1151 races across 5 eras (resets at [1966, 1989, 2014, 2026])
Engine ELO history: 9,314 rows(1950-2026)


,Engine,Rating,Rating mu,Rating sig,Races,Best perf,Last season
Rank,,,,,,,
1,BMW,2229,2229,80,251,2526,2009
2,Toyota,2192,2192,80,140,2469,2009
3,Mercedes,2062,2062,130,631,2603,2026
4,Mugen Honda,2027,2027,80,32,2263,1999
5,Honda,2005,2005,80,429,2285,2021
6,Epperly,1964,1964,94,5,2107,1960
7,Ferrari,1887,1887,130,1127,2434,2026
8,Kurtis Kraft,1878,1878,81,12,2168,1960
9,Judd,1871,1871,80,32,2120,1989


## Model Feature Dataset

Joins all data sources into a single model_df ready for model training.

**Target**: `delta_fastest` — seconds behind the fastest car on the same lap (0 for the leader, positive for everyone behind).

Lap-level features cover race state (position, gaps, pit markers), tyre state, 3-lap rolling driver pace, track conditions (flag, weather, circuit layout), and pre-race ELO ratings for constructor, engine, and driver. Feature taxonomy is defined in `FEATURE_COLS`.

In [17]:
feat = race_laps.copy()

feat["lap_time_s"] = feat["lap_time"].dt.total_seconds()
feat["cumulative_time_s"] = feat["cumulative_time"].dt.total_seconds()

feat["constructor"] = (
    feat["team_driver_id"]
    .map(team_driver_csv.set_index("id")["team_id"])
    .map(team_csv.set_index("id")["name"])
)

ctor_elo = (
    constructor_elo_history_df[["season", "round", "constructor", "rating_mu", "rating_sig"]]
    .rename(columns={"season": "year", "round": "round_number",
                     "rating_mu": "ctor_elo_mu", "rating_sig": "ctor_elo_sig"})
    .sort_values(["constructor", "year", "round_number"])
)
ctor_elo[["ctor_elo_mu", "ctor_elo_sig"]] = (
    ctor_elo.groupby("constructor")[["ctor_elo_mu", "ctor_elo_sig"]].shift(1)  
)
feat = feat.merge(ctor_elo, on=["year", "round_number", "constructor"], how="left")

unique_combos = feat[["year", "constructor"]].dropna().drop_duplicates()
engine_lookup = {
    (int(row.year), row.constructor): resolve_engine(int(row.year), row.constructor)
    for row in unique_combos.itertuples(index=False)
}
feat["engine"] = [
    engine_lookup.get((int(yr), ctor)) if pd.notna(ctor) else None
    for yr, ctor in zip(feat["year"], feat["constructor"])
]

eng_elo = (
    engine_elo_history_df[["season", "round", "engine", "rating_mu", "rating_sig"]]
    .rename(columns={"season": "year", "round": "round_number",
                     "rating_mu": "eng_elo_mu", "rating_sig": "eng_elo_sig"})
    .sort_values(["engine", "year", "round_number"])
)
eng_elo[["eng_elo_mu", "eng_elo_sig"]] = (
    eng_elo.groupby("engine")[["eng_elo_mu", "eng_elo_sig"]].shift(1)  
)
feat = feat.merge(eng_elo, on=["year", "round_number", "engine"], how="left")

feat = feat.sort_values(["year", "round_number", "driver", "number"])
feat["rolling_lap_time_3"] = (
    feat.groupby(["year", "round_number", "driver"])["lap_time_s"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# backfill tyre data within stint
feat["compound"] = feat.groupby(["year", "round_number", "driver"])["compound"].transform(
    lambda x: x.bfill()
)

def _fill_tyre_age(g):
    ta = g["tyre_age"].copy()
    laps = g["number"]
    fvi = ta.first_valid_index()
    if fvi is not None:
        first_age = ta[fvi]
        first_lap = laps[fvi]
        before = ta.isna() & (laps < first_lap)
        ta[before] = (first_age - (first_lap - laps[before])).clip(lower=1)
    return ta

feat["tyre_age"] = feat.groupby(
    ["year", "round_number", "driver"], group_keys=False
).apply(_fill_tyre_age)

feat.loc[(feat["position"] == 1) & feat["gap_from_ahead"].isna(), "gap_from_ahead"] = 0.0
_max_pos = feat.groupby(["session_id", "number"])["position"].transform("max")
feat.loc[(feat["position"] == _max_pos) & feat["gap_from_behind"].isna(), "gap_from_behind"] = 0.0


_leader_lap = (
    feat[feat["position"] == 1]
    .groupby(["session_id", "number"])["lap_time_s"]
    .first()
    .reset_index()
    .rename(columns={"lap_time_s": "_leader_lap_time_s"})
)
feat = feat.merge(_leader_lap, on=["session_id", "number"], how="left")
feat["delta_race_leader"] = feat["lap_time_s"] - feat["_leader_lap_time_s"]
feat.drop(columns=["_leader_lap_time_s"], inplace=True)


feat["delta_fastest"] = (
    feat["lap_time_s"]
    - feat.groupby(["session_id", "number"])["lap_time_s"].transform("min")
)

TARGET = "delta_fastest"
FEATURE_COLS = [
    "year", "round_number", "number", "driver", "constructor", "engine",
    "delta_fastest", "delta_race_leader",
    "position", "pit_lap", "cumulative_time_s",
    "gap_from_leader", 
    "gap_from_ahead", "gap_from_behind",
    "compound", "tyre_age",
    "rolling_lap_time_3",
    "flag", "laps_since_flag_change",
    "air_temp", "track_temp", "wind_speed", "wind_direction",
    "corners", "slow_corners", "medium_corners", "high_speed_corners",
    "ctor_elo_mu",
    "eng_elo_mu",
    "driver_elo_mu", 
]

model_df = feat[[c for c in FEATURE_COLS if c in feat.columns]].copy()
model_df = model_df.dropna(subset=[TARGET])

print(f"Model dataset: {len(model_df):,} rows x {len(model_df.columns)} columns")
print(f"Coverage: {model_df['year'].min():.0f}-{model_df['year'].max():.0f}, "
      f"{model_df['round_number'].nunique()} rounds, "
      f"{model_df['driver'].nunique()} drivers")
print(f"Target '{TARGET}': mean={model_df[TARGET].mean():.2f}s std={model_df[TARGET].std():.2f}s")
print(f"\nMissing values per column:")
missing = model_df.isnull().sum()
print(missing[missing > 0].to_string())
model_df.head()


/tmp/ipykernel_92333/3958580864.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.bfill()
/tmp/ipykernel_92333/3958580864.py:68: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(_fill_tyre_age)


Model dataset: 615,754 rows x 29 columns
Coverage: 1996-2025, 24 rounds, 146 drivers
Target 'delta_fastest': mean=4.41s std=37.28s

Missing values per column:
position                      16
pit_lap                   268833
gap_from_behind               15
compound                  427156
tyre_age                  427444
rolling_lap_time_3         11509
flag                      426729
laps_since_flag_change    426729
air_temp                  426749
track_temp                426749
wind_speed                426749
wind_direction            426749
corners                   426643
slow_corners              426643
medium_corners            426643
high_speed_corners        426643
ctor_elo_mu                 2012
eng_elo_mu                   539


,year,round_number,number,driver,constructor,engine,delta_fastest,delta_race_leader,position,pit_lap,...,air_temp,track_temp,wind_speed,wind_direction,corners,slow_corners,medium_corners,high_speed_corners,ctor_elo_mu,eng_elo_mu
0,1996,1,1.0,alesi,Benetton,Renault,2.804,2.804,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2331.0,2023.0
1,1996,1,2.0,alesi,Benetton,Renault,0.518,0.518,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2331.0,2023.0
2,1996,1,3.0,alesi,Benetton,Renault,1.071,0.871,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2331.0,2023.0
3,1996,1,4.0,alesi,Benetton,Renault,0.970,0.789,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2331.0,2023.0
4,1996,1,5.0,alesi,Benetton,Renault,0.542,0.542,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2331.0,2023.0


## Driver ELO

Rates every driver from 1950-2025 using pairwise Elo. Each race is
decomposed into N*(N-1)/2 pairwise match-ups (driver who finished ahead wins).

The library's built-in MOV multiplier (designed for NBA point differences) is disabled via a thin subclass, giving plain win/loss Elo with K=20. Ratings reset and compress toward 1500 at major regulation changes (2014, 2022).

In [18]:
from elopy.elo import Elo

DRIVER_RESET_YEARS = [2014, 2022]
DRIVER_RESET_FACTOR = 0.5
DRIVER_BASE_RATING = 1500
DRIVER_K = 20 

class _DriverElo(Elo):
    @staticmethod
    def _mov_multiplier(mov, elo_diff):
        return 1.0

driver_base = race_session_entries[["year", "round_number", "team_driver_id", "position"]].copy()
driver_base["driver"] = (
    driver_base["team_driver_id"]
    .map(team_driver_csv.set_index("id")["driver_id"])
    .map(driver_csv.set_index("id")["reference"])
)
driver_base = driver_base.dropna(subset=["driver", "year", "round_number"]).copy()
driver_base["year"] = driver_base["year"].astype(int)
driver_base["round_number"] = driver_base["round_number"].astype(int)

if extra_driver_base_rows:
    extra_db = pd.DataFrame(extra_driver_base_rows)[
        ["year", "round_number", "driver", "position"]
    ]
    driver_base = pd.concat([driver_base, extra_db], ignore_index=True)
    print(f"Appended {len(extra_db)} FastF1 driver-result rows to driver_base "
          f"({extra_db[['year','round_number']].drop_duplicates().shape[0]} races)")

all_driver_race_results = {
    (int(s), int(r)): g[["driver", "position"]].reset_index(drop=True)
    for (s, r), g in driver_base.groupby(["year", "round_number"])
}
sorted_driver_results = sorted(all_driver_race_results.items())

era_splits_d = split_into_eras(sorted_driver_results, DRIVER_RESET_YEARS)

driver_elo_history_rows = []
carry_ratings: dict[str, float] = {}

for era_i, era_items in enumerate(era_splits_d):
    if not era_items:
        continue

    era_drivers = {
        d for _, df in era_items for d in df["driver"].dropna().unique()
    }

    if era_i > 0:
        carry_ratings = {
            name: DRIVER_BASE_RATING + (r - DRIVER_BASE_RATING) * DRIVER_RESET_FACTOR
            for name, r in carry_ratings.items()
            if name in era_drivers
        }

    driver_elos: dict[str, _DriverElo] = {
        name: _DriverElo(start_elo=rating, k=DRIVER_K, hca=0)
        for name, rating in carry_ratings.items()
    }

    for (season, _round), race_df in era_items:
        # add any driver appearing for the first time
        for driver in race_df["driver"].dropna().unique():
            if driver not in driver_elos:
                driver_elos[driver] = _DriverElo(start_elo=DRIVER_BASE_RATING, k=DRIVER_K, hca=0)

        pos = pd.to_numeric(race_df["position"], errors="coerce")
        finished = (
            race_df.assign(pos_num=pos)
            .dropna(subset=["pos_num"])
            .sort_values("pos_num")
        )
        drivers_sorted = finished["driver"].dropna().tolist()

        for i, winner in enumerate(drivers_sorted):
            for j in range(i + 1, len(drivers_sorted)):
                loser = drivers_sorted[j]
                driver_elos[winner].play_game(driver_elos[loser], point_difference=1, is_home=True)

        for driver in drivers_sorted:
            driver_elo_history_rows.append({
                "season": season,
                "round": _round,
                "driver": driver,
                "rating_mu": round(driver_elos[driver].elo, 2),
            })

    carry_ratings = {
        name: obj.elo
        for name, obj in driver_elos.items()
        if name in era_drivers
    }

n_eras_d = len([e for e in era_splits_d if e])
print(f"Driver ELO: {len(sorted_driver_results)} races, {n_eras_d} eras"
      f"(resets at {DRIVER_RESET_YEARS}), K = {DRIVER_K}")

driver_elo_history_df = (
    pd.DataFrame(driver_elo_history_rows)
    .sort_values(["season", "round", "driver"])
    .reset_index(drop=True)
)
print(f"Driver ELO history: {len(driver_elo_history_df):,} rows"
      f"({driver_elo_history_df['season'].min()}-{driver_elo_history_df['season'].max()})")

driver_elo_summary_df = (
    driver_elo_history_df
    .sort_values(["driver", "season", "round"])
    .groupby("driver")[["season", "round", "rating_mu"]]
    .last()
    .rename(columns={"rating_mu": "Rating", "season": "Last season", "round": "Last round"})
    .sort_values("Rating", ascending=False)
    .reset_index()
)
driver_elo_summary_df.index += 1
driver_elo_summary_df.index.name = "Rank"
display(driver_elo_summary_df.head(20))

drv_elo = (
    driver_elo_history_df[["season", "round", "driver", "rating_mu"]]
    .rename(columns={"season": "year", "round": "round_number",
                     "rating_mu": "driver_elo_mu"})
    .sort_values(["driver", "year", "round_number"])
)
drv_elo["driver_elo_mu"] = drv_elo.groupby("driver")["driver_elo_mu"].shift(1)  
model_df = model_df.merge(drv_elo, on=["year", "round_number", "driver"], how="left")

print(f"\nmodel_df: {len(model_df):,} rows x {len(model_df.columns)} columns")
print(f"driver_elo_mu non-null: {model_df['driver_elo_mu'].notna().sum():,} / {len(model_df):,}")

final = model_df[model_df["year"] >= 2018].copy()
final.to_csv("./export/f1_model_dataset.csv", index=False)
print(f"Re-exported f1_model_dataset.csv ({len(final):,} rows x {len(final.columns)} columns)")


Appended 44 FastF1 driver-result rows to driver_base (2 races)
Driver ELO: 1151 races, 3 eras(resets at [2014, 2022]), K = 20
Driver ELO history: 25,917 rows(1950-2026)


,driver,Last season,Last round,Rating
Rank,,,,
1,rosberg,2016,21,2247.20
2,russell,2026,2,2061.05
3,prost,1993,16,2008.89
4,webber,2013,19,1969.26
5,clark,1968,1,1966.69
6,antonelli,2026,2,1946.16
7,hawthorn,1958,11,1942.98
8,rathmann,1960,3,1934.37
9,leclerc,2026,2,1933.06



model_df: 615,754 rows x 30 columns
driver_elo_mu non-null: 610,339 / 615,754
Re-exported f1_model_dataset.csv (189,111 rows x 30 columns)


## FastF1 Model Supplement

Builds lap-level rows from `ff1Cache` for races not in Jolpica sessionentry and appends them to `model_df`, then re-exports the final CSV.

In [19]:
def _laps_since_flag_change(flags):
    out, count, prev = [], 0, None
    for f in flags:
        if f != prev:
            count, prev = 0, f
        out.append(count)
        count += 1
    return out

def _safe(row, col, default=float("nan")):
    v = getattr(row, col, None)
    return float(v) if v is not None and pd.notna(v) else default

def _build_elo_map(history_df, entity_col, elo_col):
    # build pre_race elo lookup. shift_1 makes each entry the pre_race value
    h = (
        history_df[["season", "round", entity_col, "rating_mu"]]
        .rename(columns={"season": "year", "round": "round_number", "rating_mu": elo_col})
        .sort_values([entity_col, "year", "round_number"])
    )
    h[elo_col] = h.groupby(entity_col)[elo_col].shift(1)
    h[["year", "round_number"]] = h[["year", "round_number"]].astype(int)
    return h.set_index(["year", "round_number", entity_col])[elo_col].to_dict()

ctor_elo_map = _build_elo_map(constructor_elo_history_df, "constructor", "ctor_elo_mu")
eng_elo_map = _build_elo_map(engine_elo_history_df, "engine", "eng_elo_mu")
drv_elo_map = _build_elo_map(driver_elo_history_df, "driver", "driver_elo_mu")

# Fallback medians / brand_new entities with no ELO history get median
_med_ctor = model_df["ctor_elo_mu"].median()
_med_eng = model_df["eng_elo_mu"].median()
_med_drv = model_df["driver_elo_mu"].median()

existing_pairs = set(zip(model_df["year"].astype(int), model_df["round_number"].astype(int)))
missing_pairs = sorted(
    set(zip(ff1_cache["year"].astype(int), ff1_cache["round_number"].astype(int)))
    - existing_pairs
) if not ff1_cache.empty else []

print(f"Races in ff1_cache but missing from model_df: {missing_pairs}")

extra_rows: list[dict] = []

abbr_to_ref = (
    driver_csv.dropna(subset=["abbreviation"])
    .set_index("abbreviation")["reference"]
    .to_dict()
)

for (yr, rnd) in missing_pairs:
    print(f"Building rows for {yr} R{rnd} from FastF1 ...")

    session = _ff1_extra_sessions.get((yr, rnd))
    if session is None:
        try:
            session = fastf1.get_session(yr, rnd, "R")
            session.load(laps=True, telemetry=False, weather=False, messages=False)
        except Exception as exc:
            print(f"Skipped (FastF1 load failed): {exc}")
            continue

    ff1_race = ff1_cache[(ff1_cache["year"] == yr) & (ff1_cache["round_number"] == rnd)].copy()
    if ff1_race.empty:
        print("Skipped (no ff1_cache rows)")
        continue

    try:
        circuit_name = session.event["EventName"]
        corner_row = corners_csv[corners_csv["circuit"].str.lower() == circuit_name.lower()]
        if corner_row.empty:
            first_token = circuit_name.split()[0].lower()
            corner_row = corners_csv[corners_csv["circuit"].str.lower().str.startswith(first_token)]
        n_corners = int(corner_row["total_corners"].iloc[0]) if not corner_row.empty else 0
        n_slow = int(corner_row["slow_corners"].iloc[0]) if not corner_row.empty else 0
        n_medium = int(corner_row["medium_corners"].iloc[0]) if not corner_row.empty else 0
        n_high = int(corner_row["high_speed_corners"].iloc[0]) if not corner_row.empty else 0
    except Exception:
        n_corners = n_slow = n_medium = n_high = 0

    all_laps = session.laps.copy()
    all_laps["lap_time_s"] = all_laps["LapTime"].dt.total_seconds()
    all_valid = (
        all_laps[all_laps["lap_time_s"].notna()]
        .sort_values(["LapNumber", "Driver"])
        .copy()
    )
    all_valid["cumulative_time_s"] = all_valid.groupby("Driver")["lap_time_s"].cumsum()
    all_valid = all_valid.sort_values(["LapNumber", "cumulative_time_s"]).reset_index(drop=True)
    all_valid["delta_fastest"] = (
        all_valid["lap_time_s"]
        - all_valid.groupby("LapNumber")["lap_time_s"].transform("min")
    )
    all_valid["delta_race_leader"] = (
        all_valid["cumulative_time_s"]
        - all_valid.groupby("LapNumber")["cumulative_time_s"].transform("min")
    )
    all_valid["gap_from_leader"] = all_valid["delta_race_leader"]
    all_valid["gap_from_ahead"] = all_valid.groupby("LapNumber")["cumulative_time_s"].diff()
    all_valid["gap_from_behind"] = -all_valid.groupby("LapNumber")["cumulative_time_s"].diff(-1)

    for abbr, grp in all_valid.groupby("Driver"):
        grp = grp.sort_values("LapNumber").reset_index(drop=True)
        drv_ref = abbr_to_ref.get(abbr, abbr.lower())

        try:
            drv_info = session.get_driver(abbr)
            team_name = str(drv_info.get("TeamName", "UNKNOWN"))
        except Exception:
            team_name = "UNKNOWN"

        engine_name = resolve_engine(yr, team_name) or "UNKNOWN"

        pit_in = set(grp[grp["PitInTime"].notna()]["LapNumber"].values if "PitInTime" in grp.columns else [])
        pit_out = set(grp[grp["PitOutTime"].notna()]["LapNumber"].values if "PitOutTime" in grp.columns else [])

        flags = list(
            (grp["TrackStatus"].apply(parse_flag)
             if "TrackStatus" in grp.columns
             else pd.Series(["CLEAR"] * len(grp))).values
        )
        rolling_lt3 = grp["lap_time_s"].shift(1).rolling(3, min_periods=1).mean().values
        lsfc = _laps_since_flag_change(flags)

        for i, row in enumerate(grp.itertuples()):
            lap_n = int(row.LapNumber)
            pit_lp = 1 if lap_n in pit_in else (2 if lap_n in pit_out else 0)

            extra_rows.append({
                "year": yr,
                "round_number": rnd,
                "number": lap_n,
                "driver": drv_ref,
                "constructor": team_name,
                "engine": engine_name,
                "delta_fastest": float(row.delta_fastest),
                "delta_race_leader": float(row.delta_race_leader),
                "position": _safe(row, "Position"),
                "pit_lap": pit_lp,
                "cumulative_time_s": float(row.cumulative_time_s),
                "gap_from_leader": float(row.gap_from_leader),
                "gap_from_ahead": _safe(row, "gap_from_ahead"),
                "gap_from_behind": _safe(row, "gap_from_behind"),
                "compound": str(row.Compound).upper() if hasattr(row, "Compound") and pd.notna(row.Compound) else "UNKNOWN",
                "tyre_age": _safe(row, "TyreLife"),
                "rolling_lap_time_3": float(rolling_lt3[i]),
                "flag": flags[i],
                "laps_since_flag_change": lsfc[i],
                "air_temp": _safe(row, "AirTemp", 25.0),
                "track_temp": _safe(row, "TrackTemp", 30.0),
                "wind_speed": _safe(row, "WindSpeed", 0.0),
                "wind_direction": _safe(row, "WindDirection", 0.0),
                "corners": n_corners,
                "slow_corners": n_slow,
                "medium_corners": n_medium,
                "high_speed_corners": n_high,
                "ctor_elo_mu": ctor_elo_map.get((yr, rnd, team_name), _med_ctor),
                "eng_elo_mu": eng_elo_map.get((yr, rnd, engine_name), _med_eng),
                "driver_elo_mu": drv_elo_map.get((yr, rnd, drv_ref), _med_drv),
            })

    n_added = sum(1 for r in extra_rows if r["year"] == yr and r["round_number"] == rnd)
    print(f"Added {n_added} rows")

if extra_rows:
    extra_df = pd.DataFrame(extra_rows)
    shared_cols = [c for c in model_df.columns if c in extra_df.columns]
    extra_df = extra_df[shared_cols]
    model_df = pd.concat([model_df, extra_df], ignore_index=True)
    model_df = model_df.sort_values(["year", "round_number", "driver", "number"]).reset_index(drop=True)

    final = model_df[model_df["year"] >= 2018].copy()
    final.to_csv("./export/f1_model_dataset.csv", index=False)
    print(f"\nmodel_df after FastF1 supplement: {len(model_df):,} rows")
    print(f"Re-exported f1_model_dataset.csv ({len(final):,} rows x {len(final.columns)} columns)")
else:
    print("Nothing to supplement model_df already up to date.")


Races in ff1_cache but missing from model_df: [(2026, 1), (2026, 2)]
Building rows for 2026 R1 from FastF1 ...
Added 1000 rows
Building rows for 2026 R2 from FastF1 ...
Added 904 rows

model_df after FastF1 supplement: 617,658 rows
Re-exported f1_model_dataset.csv (191,015 rows x 30 columns)


## Export Datasets

Saves five CSVs used downstream for model training and analysis.

In [20]:
EXPORT_PATH = "./export"
EXPORT_COLS_LAPS = [
    "year", "round_number", "number", "position", "driver",
    "lap_time_s", "cumulative_time_s",
    "gap_from_leader", "gap_from_ahead", "gap_from_behind",
    "pit_lap",
    "compound", "tyre_age",
    "flag", "laps_since_flag_change",
    "speed_trap", "air_temp", "track_temp", "wind_speed", "wind_direction",
    "sector1_time", "sector2_time", "sector3_time",
    "corners", "slow_corners", "medium_corners", "high_speed_corners",
]

export_laps = race_laps.copy()
export_laps["lap_time_s"] = export_laps["lap_time"].dt.total_seconds()
export_laps["cumulative_time_s"] = export_laps["cumulative_time"].dt.total_seconds()

lap_export_cols = [c for c in EXPORT_COLS_LAPS if c in export_laps.columns]

export_laps[lap_export_cols].to_csv(f"{EXPORT_PATH}/export_laps.csv", index=False)
corners_csv.to_csv(f"{EXPORT_PATH}/export_circuit_corners.csv", index=False)
constructor_elo_history_df.to_csv(f"{EXPORT_PATH}/export_constructor_elo.csv", index=False)
engine_elo_history_df.to_csv(f"{EXPORT_PATH}/export_engine_elo.csv", index=False)
driver_elo_history_df.to_csv(f"{EXPORT_PATH}/export_driver_elo.csv", index=False)
print("Done")

Done
